Imported Libraries for Data Handling, model preporation and evaluating, and machine Learning 

In [23]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    balanced_accuracy_score,
    classification_report,
)
from sklearn.preprocessing import LabelEncoder

import xgboost as xgb


Loading and Preparing the physical Dataset 
    1: load each csv file
        -use pandas.read_csv() and parse the bucket column as a datetime since it represents the timestamp of each measurement.
         This ensures proper time alignment later when combining with network data.
    2: add a agg_window colum
        -Each dataset is tagged with the aggregation window it came from (10s, 16s, or 30s).
         This provides useful metadata and helps distinguish rows originating from different sampling frequencies. (this is what the paper does)
    3: concatinate all physical data into a single dataframe.
    4: inspect all colums and check for duplicates
        -print the set of columns present in each physical file (to confirm schema consistency)
        -print the number of duplicate timestamps in the combined physical dataset
        -Duplicate timestamps are expected because many sensors report measurements at the same moment.
         These will be resolved later

In [24]:
phy10s = pd.read_csv(
    r"C:\Users\amira\CISProj\CIS\exports\data\phys_agg_10s.csv",
    parse_dates=["bucket"],
)
phy16s = pd.read_csv(
    r"C:\Users\amira\CISProj\CIS\exports\data\phys_agg_16s.csv",
    parse_dates=["bucket"],
)
phy30s = pd.read_csv(
    r"C:\Users\amira\CISProj\CIS\exports\data\phys_agg_30s.csv",
    parse_dates=["bucket"],
)

phy10s["agg_window"] = 10
phy16s["agg_window"] = 16
phy30s["agg_window"] = 30

phyDf = pd.concat([phy10s, phy16s, phy30s], ignore_index=True)

print("Physical columns per file:")
print(set(phy10s.columns))
print(set(phy16s.columns))
print(set(phy30s.columns))

print("Physical duplicate buckets:", phyDf["bucket"].duplicated().sum())


Physical columns per file:
{'agg_window', 'asset_id', 'attack_types', 'system_id', 'num_attacks', 'min_value', 'avg_value', 'prop_key', 'num_measurements', 'max_value', 'bucket'}
{'agg_window', 'asset_id', 'attack_types', 'system_id', 'num_attacks', 'min_value', 'avg_value', 'prop_key', 'num_measurements', 'max_value', 'bucket'}
{'agg_window', 'asset_id', 'attack_types', 'system_id', 'num_attacks', 'min_value', 'avg_value', 'prop_key', 'num_measurements', 'max_value', 'bucket'}
Physical duplicate buckets: 84468


Loading and preparing network datasets 
    1: Load each newtork csv
        -We read each file using pandas.read_csv() and parse the bucket column as a datetime value.
         This ensures proper time-based alignment when combining physical and network layers.
    2: add a agg_window colum 
        -Each dataset is tagged with its aggregation interval (10s, 16s, or 30s).
         This allows the model to learn patterns that may vary by sampling frequency.
    3: concatinate all 3 datasets into one
    4: inspect colums for duplicates 
        -Counting duplicate timestamps reveals how many rows share the same time bucket, which is expected in network data where several devices may     communicate simultaneously.
     

In [25]:
network10s = pd.read_csv(
    r"C:\Users\amira\CISProj\CIS\exports\data\scada_resolved_agg_10s.csv",
    parse_dates=["bucket"],
)
network16s = pd.read_csv(
    r"C:\Users\amira\CISProj\CIS\exports\data\scada_resolved_agg_16s.csv",
    parse_dates=["bucket"],
)
network30s = pd.read_csv(
    r"C:\Users\amira\CISProj\CIS\exports\data\scada_resolved_agg_30s.csv",
    parse_dates=["bucket"],
)

network10s["agg_window"] = 10
network16s["agg_window"] = 16
network30s["agg_window"] = 30

networkDf = pd.concat([network10s, network16s, network30s], ignore_index=True)

print("\nNetwork columns per file:")
print(set(network10s.columns))
print(set(network16s.columns))
print(set(network30s.columns))

print("Network duplicate buckets:", networkDf["bucket"].duplicated().sum())



Network columns per file:
{'agg_window', 'num_attacks', 'destination_asset', 'min_size', 'system_id', 'source_mac', 'destination_port', 'bucket', 'avg_size', 'source_asset', 'destination_mac', 'attack_types', 'num_connections', 'max_size', 'protocol', 'source_ip', 'destination_key', 'source_key', 'destination_ip', 'source_port', 'destination_total_packets', 'source_total_packets'}
{'agg_window', 'num_attacks', 'destination_asset', 'min_size', 'system_id', 'source_mac', 'destination_port', 'bucket', 'avg_size', 'source_asset', 'destination_mac', 'attack_types', 'num_connections', 'max_size', 'protocol', 'source_ip', 'destination_key', 'source_key', 'destination_ip', 'source_port', 'destination_total_packets', 'source_total_packets'}
{'agg_window', 'num_attacks', 'destination_asset', 'min_size', 'system_id', 'source_mac', 'destination_port', 'bucket', 'avg_size', 'source_asset', 'destination_mac', 'attack_types', 'num_connections', 'max_size', 'protocol', 'source_ip', 'destination_key',

Cleaning physical data from all duplicates to get one snapshot per timestamp 
    1: select nummaric measured physical features from dataset
        ex: min_value – minimum observed sensor value in the aggregation window
        -These describe the physical behavior of the system over time, independent of specific assets or sensors.
    2: aggergate to one row per bucket 
        -groupby("bucket") collects all physical rows that share the same timestamp.
        -mean() computes the average of each numeric physical feature across all sensors/props at that time.
        -reset_index() returns a clean DataFrame where each row is one physical snapshot = one timestamp
        -After this step, phyAgg contains exactly one row per timestamp, summarizing the overall physical state of the system.
    

In [26]:
phys_numeric_cols = ["min_value", "max_value", "avg_value", "num_measurements"]

phyAgg = (
    phyDf
    .groupby("bucket")[phys_numeric_cols]
    .mean()
    .reset_index()
)


Combining physical and network data with time alignment
 -Since physical data is sampled less frequently than network traffic, we use a time-aware merge to attach the correct physical snapshot to each network record.
    1: ensure timestamps are in a datetime format
    2: Time-align physical and network data (core of the paper’s method)
        -merge_asof performs a time-aware join, not a normal merge. It aligns each network row with the closest physical snapshot at or before the   network timestamp
        -direction="backward" means: “For each network timestamp, attach the most recent earlier physical state.”
        - the is  academic paper’s description of: “concatenating the most recent anterior physical data to each network data entry.”

In [27]:
networkDf["bucket"] = pd.to_datetime(networkDf["bucket"])
phyAgg["bucket"]    = pd.to_datetime(phyAgg["bucket"])

combinedDf = pd.merge_asof(
    networkDf.sort_values("bucket"),
    phyAgg.sort_values("bucket"),
    on="bucket",
    direction="backward"
)


Cleaningn and normalizing attack Labels
 - the attack_types in physical and network data use encoded strings like 'normal' 'alnomaly' and 'MITM'. These values are not immediately usable as machine-learning labels.
    1: Parses the attack_types field
        -The original column stores labels like "['DoS', 'anomaly']" as strings, not lists.
        -We convert each string into a real Python list
    2: normalize labels to lowercase
    3: Resolves multilabel rows into ONE attack class
        -Some rows contain multiple labels ex "['DoS', 'anomaly']".
        so i mapped these to a single class using this prority 
            -Scan   
             DoS
             MITM
             Physical Fault
             Normal
             Anomaly
        - this creates something clean that the XGboost can really use and learn on 
    4: Produce a new unified target label column
        -This becomes the final label used for training and evaluating the model.

In [28]:
import ast

def map_attack_types(s):
    if pd.isna(s):
        return "Normal"
    try:
        labels = ast.literal_eval(s)
    except:
        labels = [str(s)]

    labels = [lbl.lower() for lbl in labels]

    if "scan" in labels:
        return "Scan"
    if "dos" in labels:
        return "DoS"
    if "mitm" in labels:
        return "MITM"
    if "physical fault" in labels:
        return "Physical Fault"
    if "normal" in labels:
        return "Normal"
    if "anomaly" in labels:
        return "Anomaly"

    return labels[0].title()

combinedDf["attack_class"] = combinedDf["attack_types"].apply(map_attack_types)


Selecting final attack classes used for classification 
    -the paper only used 5 classification types, and assuming we want to do the same i filtered the dataset to include only these five classes.
    1: define the target label sets
        -matches the classes reported in the research paper.
    2: filters the dataset 
        -This keeps only rows whose attack_class is one of the five categories.

In [29]:
valid_classes = ["Normal", "DoS", "MITM", "Physical Fault", "Scan"]
combinedDf_filtered = combinedDf[combinedDf["attack_class"].isin(valid_classes)].copy()


Building Feature Matrix (X) and Target Labels (y)
    1: select the target label
        -The cleaned and standardized attack_class column becomes the target output we want the model to predict.
    2: Remove columns that should not be used as features
        - i dropped colums that leak label information, Columns that are identifiers, not meaningful features
        - removing them helps stop overfitting, memorization of device identities, poor generalization
        -Removing them forces the model to learn from behavior, not IDs.
    3: Identify categorical columns
    4: Encode categorical features numerically
        - ML models like xgboost cannot use strings, so i converted those strings into ints
        -A dictionary of encoders is stored so we can translate values back later if needed.


In [ ]:
y = combinedDf_filtered["attack_class"]

drop_cols = [
    "bucket", "attack_types", "attack_class", "num_attacks",
    "source_ip", "destination_ip", "source_mac", "destination_mac",
    "source_key", "destination_key", "system_id"
]

X = combinedDf_filtered.drop(columns=[c for c in drop_cols if c in combinedDf_filtered.columns])


cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le


Encoding the Target Labels for Machine Learning
 -before training xgboost, attack_types must be converted to ints 
    1: create a encoder
        -This encoder maps each unique attack class to an integer ID.
    2: Fit the encoder and transform the labels
        - ex: DoS             → 0  
              MITM            → 1  
              Normal          → 2  
              Physical Fault  → 3  
              Scan            → 4
        -The result, y_encoded, is a NumPy array of integers that XGBoost can train on.

In [31]:
le_y = LabelEncoder()
y_encoded = le_y.fit_transform(y)

print("Label classes:", le_y.classes_)


Label classes: ['DoS' 'MITM' 'Normal' 'Physical Fault' 'Scan']


Splitting the Dataset into Training and Testing Sets
    1: create a 80/20 split 
        -80% for training
        -20% for testing
    2: Stratify the split by class
        -This is extremely important because CPS datasets are imbalanced (many Normal events, far fewer attacks).
         Stratification ensures that both the training and test sets contain the same class distribution
         - i read this in a  'best practices used in academic ML experiments' article i found online.
    3: Set a random seed for reproducibility

In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42,
)


Training the XGBoost Classifier 
    -What each hyperparameter does

-n_estimators=600
Number of boosted trees in the ensemble.
More trees= higher capacity= better performance (up to a point).

-max_depth=8
Maximum depth of each decision tree.
Deeper trees can model complex patterns (useful for CPS attack behavior).

-learning_rate=0.05
Shrinks each tree’s contribution.
Low values = slower but more stable learning.

-subsample=0.9
Randomly samples 90% of rows per tree.
Helps prevent overfitting.

-colsample_bytree=0.8
Randomly samples 80% of features per tree.
Further reduces overfitting and improves generalization.

-objective="multi:softmax"
Specifies this is a multi-class classification problem.
Model outputs class IDs directly.

-eval_metric="logloss"
Uses multiclass log-loss as the optimization metric.

-n_jobs=-1
Uses all CPU cores for faster training

    1: trainin the model
    2: make predictions
        -After training, the model predicts the encoded attack classes for the 20% test split.

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=600,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.8,
    objective="multi:softmax",
    eval_metric="logloss",
    n_jobs=-1,
)

xgb_model.fit(X_train, y_train)

y_pred_encoded = xgb_model.predict(X_test)


Model Evaluation: Balanced Accuracy, Classification Report, and Per-Class TPR/FPR

    1: Convert Encoded Labels Back to Human-Readable Form
        -During training, labels were encoded as integers (0–4).
        -we convert them back to the original strings
    2: Balanced Accuracy as used in the academic paper
        -This metric is the same metric used in the academic paper
        -Balanced accuracy prevents the model from cheating by always guessing Normal.
    3: Classification Report prints 
        -precision (When the model raises an alert, is it correct?)
        -recall (Of all the true attacks, how many did the model catch?)
        -f1-score (Balance between precision and recall) (F1 = 2 * (precision * recall) / (precision + recall))
        -support (sample count) - (How many examples of each class exist?)
    4: Confusion Matrix Normalized
    5: Per-Class TPR and FPR
        -The paper reports True Positive Rate (TPR) and False Positive Rate (FPR) for each attack type. so i replicated this
        -This allows direct comparison with Table 1 and Table 2 from the academic paper.

In [ ]:

y_test_labels = le_y.inverse_transform(y_test)
y_pred_labels = le_y.inverse_transform(y_pred_encoded)

bal_acc = balanced_accuracy_score(y_test_labels, y_pred_labels)
print(f"Balanced Accuracy: {bal_acc:.4f}\n")

print("Classification report:")
print(classification_report(y_test_labels, y_pred_labels))

labels = le_y.classes_
cm = confusion_matrix(y_test_labels, y_pred_labels, labels=labels)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

print("\nPer-class TPR / FPR:")
for lbl, row in zip(labels, cm_norm):
    tpr = row[list(labels).index(lbl)]  
    fpr = 1 - tpr
    print(f"{lbl:15s} TPR={tpr:.4f}  FPR={fpr:.4f}")


Balanced Accuracy: 0.9604

Classification report:
                precision    recall  f1-score   support

           DoS       1.00      1.00      1.00     93836
          MITM       0.96      0.92      0.94     11674
        Normal       0.98      0.99      0.99     98382
Physical Fault       0.93      0.90      0.91      7664
          Scan       1.00      1.00      1.00        17

      accuracy                           0.99    211573
     macro avg       0.97      0.96      0.97    211573
  weighted avg       0.99      0.99      0.99    211573


Per-class TPR / FPR:
DoS             TPR=0.9997  FPR=0.0003
MITM            TPR=0.9163  FPR=0.0837
Normal          TPR=0.9905  FPR=0.0095
Physical Fault  TPR=0.8954  FPR=0.1046
Scan            TPR=1.0000  FPR=0.0000
